# Gutenberg corpus selection for Linger

- Exclusively English, public-domain plain text only.   
- Three edit points, marked **Edit 1–3**: search parameters, candidate shortlist, and final selection.   
- Every other cell just queries and displays. 
- Filling `FINAL_SELECTION` writes a content-versioned manifest under `data/gutenberg/`.


# Imports & Config

In [1]:
from datetime import datetime, timezone
from hashlib import sha256
from io import BytesIO
import json
import math
from pathlib import Path
import re
import textwrap
import polars as pl
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://gutendex.com/books/"
CATALOG_URL = "https://www.gutenberg.org/cache/epub/feeds/pg_catalog.csv"
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "gutenberg"
FIXED_PARAMS = {"languages": "en", "mime_type": "text/plain", "copyright": "false"}
DOWNLOAD_METADATA_SCHEMA_VERSION = 1
REQUEST_TIMEOUT = (10, 90)  # seconds: connect, read

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "linger-gutenberg-notebook/2.0"})
SESSION.mount(
    "https://",
    HTTPAdapter(
        max_retries=Retry(
            total=4,
            backoff_factor=1.0,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=frozenset({"GET"}),
        )
    ),
)


def request_json(url: str, params: dict | None = None) -> dict:
    response = SESSION.get(url, params=params, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()


def join_names(people: list[dict]) -> str:
    return "; ".join(person["name"] for person in people)


def choose_text_url(book: dict) -> str | None:
    formats = book["formats"]
    preferred = (
        "text/plain; charset=utf-8",
        "text/plain; charset=us-ascii",
        "text/plain",
    )
    for mime_type in preferred:
        if formats.get(mime_type):
            return formats[mime_type]
    return next(
        (url for mime, url in formats.items() if mime.startswith("text/plain") and url),
        None,
    )


def download_book(book: dict) -> tuple[Path, dict]:
    """Return a verified cached text and provenance captured at download time."""
    requested_url = choose_text_url(book)
    if requested_url is None:
        raise ValueError(f"No plain-text file for PG #{book['id']}")

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    path = DATA_DIR / f"pg{book['id']}.txt"
    metadata_path = path.with_suffix(".metadata.json")
    metadata = None
    if path.exists() and metadata_path.exists():
        try:
            candidate_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
            actual_sha256 = sha256(path.read_bytes()).hexdigest()
            required_keys = {
                "requested_url",
                "resolved_url",
                "downloaded_at_utc",
                "sha256",
            }
            if (
                isinstance(candidate_metadata, dict)
                and candidate_metadata.get("schema_version")
                == DOWNLOAD_METADATA_SCHEMA_VERSION
                and candidate_metadata.get("gutenberg_id") == book["id"]
                and required_keys <= candidate_metadata.keys()
                and candidate_metadata["sha256"] == actual_sha256
            ):
                metadata = candidate_metadata
        except (json.JSONDecodeError, OSError):
            pass

    if metadata is None:
        response = SESSION.get(requested_url, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        path.write_bytes(response.content)
        metadata = {
            "schema_version": DOWNLOAD_METADATA_SCHEMA_VERSION,
            "gutenberg_id": book["id"],
            "requested_url": requested_url,
            "resolved_url": response.url,
            "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
            "sha256": sha256(response.content).hexdigest(),
        }
        metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    return path, metadata


START_MARKER_RE = re.compile(
    r"\*{3}\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*{3}", re.IGNORECASE
)
END_MARKER_RE = re.compile(
    r"\*{3}\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*{3}", re.IGNORECASE
)
HEADING_RE = re.compile(
    r"(?im)^\s*((?:chapter|book|part|section|canto|adventure|letter|story|act|scene)\s+"
    r"(?:[ivxlcdm]+|\d+|[a-z]+)\b[^\n]*)$"
)


def extract_book_body(path: Path) -> str:
    raw = path.read_text(encoding="utf-8-sig", errors="replace").replace("\r\n", "\n")
    start_marker = START_MARKER_RE.search(raw)
    body_start = start_marker.end() if start_marker else 0
    end_marker = END_MARKER_RE.search(raw, body_start)
    return raw[body_start : end_marker.start() if end_marker else len(raw)].strip()


pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_width_chars(160)
pl.Config.set_tbl_rows(50)

polars.config.Config

## Topic vocabulary

Valid `topic` values, derived from the official catalog (English text entries). Browse the top of the list here, or grep it with `topic_search("memory")` in a scratch cell.


In [2]:
response = SESSION.get(CATALOG_URL, timeout=REQUEST_TIMEOUT)
response.raise_for_status()
catalog = pl.read_csv(BytesIO(response.content)).filter(
    (pl.col("Type") == "Text") & (pl.col("Language") == "en")
)

TOPICS = (
    pl.concat(
        [
            catalog.select(
                pl.col("Subjects").str.split(";").alias("topic"),
                pl.lit("subject").alias("kind"),
            ),
            catalog.select(
                pl.col("Bookshelves").str.split(";").alias("topic"),
                pl.lit("bookshelf").alias("kind"),
            ),
        ]
    )
    .explode("topic", empty_as_null=False)
    .with_columns(pl.col("topic").str.strip_chars())
    .filter(pl.col("topic").is_not_null() & (pl.col("topic") != ""))
    .group_by(["kind", "topic"])
    .len(name="book_count")
    .sort(["book_count", "topic"], descending=[True, False])
)


def topic_search(term: str, *, limit: int = 30) -> pl.DataFrame:
    return TOPICS.filter(
        pl.col("topic")
        .str.to_lowercase()
        .str.contains(term.strip().lower(), literal=True)
    ).head(limit)


print(f"{len(TOPICS):,} topic options from English catalog entries")
TOPICS.head(20)

37,009 topic options from English catalog entries


kind,topic,book_count
str,str,u32
"""bookshelf""","""Category: Novels""",18500
"""bookshelf""","""Category: British Literature""",9416
"""bookshelf""","""Category: American Literature""",7591
"""bookshelf""","""Category: Adventure""",7367
"""bookshelf""","""Category: Children & Young Adult Reading""",6346
"""bookshelf""","""Category: Biographies""",5177
"""bookshelf""","""Category: History - Modern (1750+)""",5032
"""bookshelf""","""Category: History - American""",4838
"""bookshelf""","""Category: Essays, Letters & Speeches""",4524


## Edit 1 — search parameters

The search downloads each eligible plain-text book to count its words, then shows only books within the selected range.


In [3]:
SEARCH_PARAMS = {
    "search": "",  # words in author names or titles, e.g. "jane austen"
    "topic": "",  # phrase in subjects or bookshelves, from the vocabulary above
    "sort": "popular",  # popular | ascending | descending
}
PAGES = 1  # result pages to fetch, 32 books per page
MIN_WORDS = 30_000
MAX_WORDS = 150_000

if (
    not isinstance(MIN_WORDS, int)
    or isinstance(MIN_WORDS, bool)
    or not isinstance(MAX_WORDS, int)
    or isinstance(MAX_WORDS, bool)
    or MIN_WORDS < 0
    or MAX_WORDS < MIN_WORDS
):
    raise ValueError("Require integer word limits with 0 <= MIN_WORDS <= MAX_WORDS")

In [4]:
rows = []
total_matches = 0
api_records_seen = 0
next_url: str | None = BASE_URL
query = FIXED_PARAMS | {key: value for key, value in SEARCH_PARAMS.items() if value}
for page_number in range(PAGES):
    if next_url is None:
        break
    payload = request_json(next_url, query if page_number == 0 else None)
    total_matches = payload["count"]
    api_records_seen += len(payload["results"])
    for book in payload["results"]:
        if book["languages"] != ["en"] or book.get("copyright") is not False:
            continue
        path, _ = download_book(book)
        word_count = len(extract_book_body(path).split())
        if not MIN_WORDS <= word_count <= MAX_WORDS:
            continue
        rows.append(
            {
                "id": book["id"],
                "title": book["title"],
                "authors": join_names(book["authors"]),
                "words": word_count,
                "topic": SEARCH_PARAMS["topic"],
                "subjects": "; ".join(book["subjects"]),
                "bookshelves": "; ".join(book["bookshelves"]),
                "summary": " ".join(book.get("summaries") or []),
                "downloads_30d": book["download_count"],
                "gutenberg_url": f"https://www.gutenberg.org/ebooks/{book['id']}",
            }
        )
    next_url = payload.get("next")

results = pl.DataFrame(
    rows,
    schema={
        "id": pl.Int64,
        "title": pl.String,
        "authors": pl.String,
        "words": pl.Int64,
        "topic": pl.String,
        "subjects": pl.String,
        "bookshelves": pl.String,
        "summary": pl.String,
        "downloads_30d": pl.Int64,
        "gutenberg_url": pl.String,
    },
)
print(
    f"Retrieved {len(results):,} exclusively English public-domain books with "
    f"{MIN_WORDS:,}–{MAX_WORDS:,} words "
    f"from {api_records_seen:,} API records ({total_matches:,} API matches before exact-language checking)"
)
results

Retrieved 16 exclusively English public-domain books with 30,000–150,000 words from 32 API records (61,852 API matches before exact-language checking)


id,title,authors,words,topic,subjects,bookshelves,summary,downloads_30d,gutenberg_url
i64,str,str,i64,str,str,str,str,i64,str
1342,"""Pride and Prejudice""","""Austen, Jane""",127359,"""""","""Courtship -- Fiction; Domestic fiction; England -- Fiction; Love stories; Sisters -- Fiction; Social classes -- Fiction; Young women -- Fiction""","""Best Books Ever Listings; Category: British Literature; Category: Classics of Literature; Category: Novels; Category: Romance; Harvard Classics""","""""Pride and Prejudice"" by Jane Austen is a novel published in 1813. It follows Elizabeth Bennet, who must learn to see past first impressions and hasty judgments. With five daughters and an estate that…",183505,"""https://www.gutenberg.org/ebooks/1342"""
2641,"""A Room with a View""","""Forster, E. M. (Edward Morgan)""",66599,"""""","""British -- Italy -- Fiction; England -- Fiction; Florence (Italy) -- Fiction; Humorous stories; Young women -- Fiction""","""Category: British Literature; Category: Novels; Category: Romance; Italy""","""""A Room with a View"" by E. M. Forster is a novel published in 1908. Young Lucy Honeychurch travels to Italy with her uptight cousin as chaperone, where an unexpected encounter with the unconventional …",142569,"""https://www.gutenberg.org/ebooks/2641"""
1727,"""The Odyssey: Rendered into English prose for the use of those who cannot read the original""","""Homer""",129571,"""""","""Epic poetry, Greek -- Translations into English; Homer -- Translations into English; Odysseus, King of Ithaca (Mythological character)""","""Category: Classics of Literature; Category: Mythology, Legends & Folklore; Category: Poetry; Classical Antiquity; Harvard Classics""","""""The Odyssey"" by Homer is an ancient Greek epic composed around the 8th or 7th century BC. It follows Odysseus, king of Ithaca, on his perilous ten-year journey home after the Trojan War. While he bat…",139227,"""https://www.gutenberg.org/ebooks/1727"""
65238,"""The Secret of Chimneys""","""Christie, Agatha""",74617,"""""","""Battle, Superintendent (Fictitious character) -- Fiction; Detective and mystery stories; Police -- England -- Fiction""","""Category: Adventure; Category: British Literature; Category: Crime, Thrillers and Mystery; Category: Novels; Category: Romance""","""""The Secret of Chimneys"" by Agatha Christie is a suspenseful detective novel written in the early 20th century. The story introduces the charming Anthony Cade, who finds himself embroiled in a web of …",112143,"""https://www.gutenberg.org/ebooks/65238"""
2868,"""The Green Mummy""","""Hume, Fergus""",87847,"""""","""Detective and mystery stories; Fiction""","""Category: Crime, Thrillers and Mystery; Category: Novels; Category: Romance""","""""The Green Mummy"" by Fergus Hume is a novel likely written during the late 19th century. The story revolves around a young couple, Archie Hope and Lucy Kendal, as they navigate romance against a backd…",106137,"""https://www.gutenberg.org/ebooks/2868"""
1661,"""The Adventures of Sherlock Holmes""","""Doyle, Arthur Conan""",104506,"""""","""Detective and mystery stories, English; Holmes, Sherlock (Fictitious character) -- Fiction; Private investigators -- England -- Fiction""","""Banned Books from Anne Haight's list; Category: British Literature; Category: Crime, Thrillers and Mystery; Category: Short Stories; Contemporary Reviews; Detective Fiction""","""""The Adventures of Sherlock Holmes"" by Arthur Conan Doyle is a collection of short stories first published in 1892. These twelve tales feature the legendary consulting detective Sherlock Holmes and hi…",105900,"""https://www.gutenberg.org/ebooks/1661"""
67979,"""The Blue Castle: a novel""","""Montgomery, L. M. (Lucy Maud)""",68274,"""""","""Canada -- History -- 1914-1945 -- Fiction; Choice (Psychology) -- Fiction; Love -- Fiction; Romance fiction; Self-actualization (Psychology) -- Fiction; Single women -- Fiction; Young adult fiction""","""Category: Novels; Category: Romance""","""""The Blue

## Edit 2 — shortlist

Paste promising IDs from the word-count-filtered search results. The cell profiles the cached downloads; nonzero `odd_chars` is a prompt to inspect, not reject.


In [ ]:
CANDIDATE_IDS: list[int] = [2701]  # e.g. [11, 84, 1342, 1661, 1727, 2701]
if not CANDIDATE_IDS:
    print("Paste shortlisted IDs into CANDIDATE_IDS and rerun.")
if len(CANDIDATE_IDS) != len(set(CANDIDATE_IDS)):
    raise ValueError("CANDIDATE_IDS must not contain duplicates")
records: dict[int, dict] = {}
for start in range(0, len(CANDIDATE_IDS), 32):
    batch = CANDIDATE_IDS[start : start + 32]
    payload = request_json(BASE_URL, {"ids": ",".join(map(str, batch))})
    records.update({book["id"]: book for book in payload["results"]})
missing = sorted(set(CANDIDATE_IDS) - records.keys())
if missing:
    raise ValueError(f"Not found in Gutendex: {missing}")
ineligible = [
    book_id
    for book_id, book in records.items()
    if book["languages"] != ["en"]
    or book.get("copyright") is not False
    or choose_text_url(book) is None
]
if ineligible:
    raise ValueError(
        f"Candidates must be exclusively English, public domain, and available as plain text: {sorted(ineligible)}"
    )

candidates: dict[int, dict] = {}
profiles = []
for book_id in CANDIDATE_IDS:
    book = records.get(book_id)
    if book is None:
        continue
    path, download_metadata = download_book(book)
    body = extract_book_body(path)
    headings = [" ".join(heading.split()) for heading in HEADING_RE.findall(body)]
    words = body.split()

    body_sha256 = sha256(body.encode("utf-8")).hexdigest()
    candidates[book_id] = {
        "book": book,
        "body": body,
        "body_sha256": body_sha256,
        "download_metadata": download_metadata,
        "headings": headings,
        "path": path,
    }
    profiles.append(
        {
            "id": book_id,
            "title": book["title"],
            "authors": join_names(book["authors"]),
            "words": len(words),
            "estimated_pages": math.ceil(len(words) / 300),
            "headings": len(headings),
            "first_heading": headings[0] if headings else "None detected",
            "odd_chars": body.count("\ufffd")
            + body.count("\u00c3")
            + body.count("\u00c2")
            + body.count("\u00e2\u20ac"),
            "downloads_30d": book["download_count"],
        }
    )

profiles_df = pl.DataFrame(
    profiles,
    schema={
        "id": pl.Int64,
        "title": pl.String,
        "authors": pl.String,
        "words": pl.Int64,
        "estimated_pages": pl.Int64,
        "headings": pl.Int64,
        "first_heading": pl.String,
        "odd_chars": pl.Int64,
        "downloads_30d": pl.Int64,
    },
).sort("words")

profiles_df

id,title,authors,words,estimated_pages,headings,first_heading,odd_chars,downloads_30d
i64,str,str,i64,i64,i64,str,i64,i64
2701,"""Moby Dick; Or, The Whale""","""Melville, Herman""",212796,710,301,"""CHAPTER 1. Loomings.""",0,188210


## Read samples

Four passages from every shortlisted book. While reading, judge: retrieval value (recurring themes and scenes), spoiler boundaries (do headings map to "I have read up to here"?), text quality, and corpus diversity (era, style, translation, dated framing).


In [6]:
PASSAGE_WORDS = 180

for book_id, asset in candidates.items():
    book = asset["book"]
    words = asset["body"].split()
    positions = {
        "Opening": 0,
        "One-third": max(0, len(words) // 3 - PASSAGE_WORDS // 2),
        "Two-thirds": max(0, 2 * len(words) // 3 - PASSAGE_WORDS // 2),
        "Ending": max(0, len(words) - PASSAGE_WORDS),
    }
    print("=" * 100)
    print(f"PG #{book_id}: {book['title']} \u2014 {join_names(book['authors'])}")
    print(f"Subjects: {'; '.join(book['subjects'])}")
    print("Headings:", " | ".join(asset["headings"][:8]) or "None detected")
    for label, start in positions.items():
        print(f"\n--- {label} ---\n")
        print(textwrap.fill(" ".join(words[start : start + PASSAGE_WORDS]), width=100))
    print()

PG #2701: Moby Dick; Or, The Whale — Melville, Herman
Subjects: Adventure stories; Ahab, Captain (Fictitious character) -- Fiction; Mentally ill -- Fiction; Psychological fiction; Sea stories; Ship captains -- Fiction; Whales -- Fiction; Whaling -- Fiction; Whaling ships -- Fiction
Headings: CHAPTER 1. Loomings. | CHAPTER 2. The Carpet-Bag. | CHAPTER 3. The Spouter-Inn. | CHAPTER 4. The Counterpane. | CHAPTER 5. Breakfast. | CHAPTER 6. The Street. | CHAPTER 7. The Chapel. | CHAPTER 8. The Pulpit.

--- Opening ---

MOBY-DICK; or, THE WHALE. By Herman Melville CONTENTS ETYMOLOGY. EXTRACTS (Supplied by a Sub-Sub-
Librarian). CHAPTER 1. Loomings. CHAPTER 2. The Carpet-Bag. CHAPTER 3. The Spouter-Inn. CHAPTER 4.
The Counterpane. CHAPTER 5. Breakfast. CHAPTER 6. The Street. CHAPTER 7. The Chapel. CHAPTER 8. The
Pulpit. CHAPTER 9. The Sermon. CHAPTER 10. A Bosom Friend. CHAPTER 11. Nightgown. CHAPTER 12.
Biographical. CHAPTER 13. Wheelbarrow. CHAPTER 14. Nantucket. CHAPTER 15. Chowder. CHAPTE

## Edit 3 — final selection

Map chosen IDs to a one-line reason; the cell writes a versioned manifest for the corpus build.


In [7]:
FINAL_SELECTION: dict[int, str] = {
    # 1342: "Recurring social scenes and clean chapter boundaries.",
}

if FINAL_SELECTION:
    if not 3 <= len(FINAL_SELECTION) <= 5:
        raise ValueError("FINAL_SELECTION must contain 3 to 5 books")
    unknown_ids = sorted(set(FINAL_SELECTION) - candidates.keys())
    if unknown_ids:
        raise ValueError(
            f"FINAL_SELECTION contains IDs not profiled in CANDIDATE_IDS: {unknown_ids}"
        )
    blank_reasons = sorted(
        book_id for book_id, reason in FINAL_SELECTION.items() if not reason.strip()
    )
    if blank_reasons:
        raise ValueError(
            f"Every selected book needs a reason; blank for: {blank_reasons}"
        )

    manifest_books = []
    for book_id in sorted(FINAL_SELECTION):
        asset = candidates[book_id]
        book = asset["book"]
        download_metadata = asset["download_metadata"]
        manifest_books.append(
            {
                "gutenberg_id": book_id,
                "title": book["title"],
                "authors": join_names(book["authors"]),
                "requested_url": download_metadata["requested_url"],
                "resolved_url": download_metadata["resolved_url"],
                "downloaded_at_utc": download_metadata["downloaded_at_utc"],
                "source_path": str(asset["path"].relative_to(PROJECT_ROOT)),
                "raw_sha256": download_metadata["sha256"],
                "normalized_body_sha256": asset["body_sha256"],
                "words": len(asset["body"].split()),
                "reason": FINAL_SELECTION[book_id].strip(),
            }
        )

    corpus_identity = {
        "schema_version": 1,
        "books": [
            {
                key: book[key]
                for key in (
                    "gutenberg_id",
                    "raw_sha256",
                    "normalized_body_sha256",
                    "reason",
                )
            }
            for book in manifest_books
        ],
    }
    corpus_id = sha256(
        json.dumps(corpus_identity, ensure_ascii=False, sort_keys=True).encode("utf-8")
    ).hexdigest()[:12]
    manifest = {"schema_version": 1, "corpus_id": corpus_id, "books": manifest_books}
    manifest_path = DATA_DIR / f"corpus_manifest-{corpus_id}.json"
    if manifest_path.exists():
        print(f"Manifest already exists: {manifest_path}")
    else:
        manifest_path.write_text(
            json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
        )
        print(f"Wrote {manifest_path}")
else:
    print("Fill FINAL_SELECTION with {gutenberg_id: reason} and rerun.")

Fill FINAL_SELECTION with {gutenberg_id: reason} and rerun.
